# Free-threading lab

Measures how one Mandelbrot escape-count workload scales on this machine across three
execution strategies — **GIL-on threads**, **GIL-off threads** (free-threaded CPython) and
**GIL-on processes** — all in the *same* free-threaded build, dispatched through the included
AGILAB pool engine (`agilab_pool.py`).

Adapted from the original AGILAB free-threading notebook, created September 19, 2026
(AGILAB contributors, BSD-3-Clause; see `LICENSE`).
Background: [Free-threaded CPython](https://docs.python.org/3.14/howto/free-threading-python.html).

## Stage 1 — Imports, interpreter probe and workload plan

In [ ]:
import json
import sys
from pathlib import Path

try:
    PROJECT_ROOT  # injected by the notebook runner
except NameError:
    PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = Path(PROJECT_ROOT)
if str(PROJECT_ROOT) not in [str(p) for p in sys.path]:
    sys.path.insert(0, str(PROJECT_ROOT))

import free_threading_core as core
import benchmark

interpreter, probe = core.resolve_free_threading_python()
allowance = core.effective_cpu_allowance()
WORKLOAD = {"width": 192, "height": 128, "iterations": 160}
N_WORKERS = min(4, allowance)
REPEATS = 1

print("interpreter     :", interpreter)
print("build           :", probe["version"], "| free-threaded:", probe["free_threaded_build"],
      "| Py_GIL_DISABLED:", probe["py_gil_disabled_build"])
print("effective CPUs  :", allowance)
print("workload        :", WORKLOAD, "| workers:", N_WORKERS, "| repeats:", REPEATS)
print("modes           :", [core.MODE_LABELS[m].split(" (")[0] for m in core.MODES])

## Stage 2 — Run the six real benchmark cases

Each case is a separate free-threaded child process (private environment, per-mode GIL flag,
per-mode AGILAB pool backend). `results.json` is written to the notebook's execution directory.

In [ ]:
payload = benchmark.run_benchmark(
    width=WORKLOAD["width"],
    height=WORKLOAD["height"],
    iterations=WORKLOAD["iterations"],
    workers=N_WORKERS,
    repeats=REPEATS,
    tile_rows=4,
    child_timeout=60.0,
    total_timeout=120.0,
    out_path="results.json",
)
results = payload["results"]
for case in results["cases"]:
    med = case.get("median_engine_seconds")
    print(f"{case['mode']:18s} w={case['workers']}  {case['status']:5s}  "
          f"median={med:.4f}s  backend={case.get('engine_backend')}  "
          f"width={case.get('engine_width')}")
print("total wall:", results["total_seconds"], "s")

## Stage 3 — Verify the evidence and summarise speedups

In [ ]:
on_disk = json.loads(Path("results.json").read_text())
evidence = {"results": benchmark.deterministic_evidence(payload["results"])}
assert on_disk == evidence, "results.json does not match the deterministic evidence"

digests = {case["digest"] for case in results["cases"] if case["status"] == "ok"}
assert len(digests) == 1, "modes disagree on the computed image"
assert results["same_as_serial_reference"], "pool result differs from the serial reference"
assert results["same_work"] is True

print("image digest (sha256):", results["digest"])
print("same work across all 6 cases: True")
print("matches serial reference:     True")
print()
for mode in core.MODES:
    entry = results["speedups"][mode]
    print(f"{core.MODE_LABELS[mode].split(' (')[0]:18s}  1w={entry['baseline_median_seconds']:.4f}s  "
          f"{entry['workers']}w={entry['wide_median_seconds']:.4f}s  "
          f"speedup={entry['speedup']:.2f}x")
print()
print("evidence written to:", Path("results.json").resolve())